# Phase 1: Synthetic MDPs and Ground Truth

## Random Search Driven Composite Neural Architecture Search for Multi-Source RL State Encoding

This notebook implements Phase 1 of the project:
- Task 1.1: Simple Synthetic MDP
- Task 1.2: Ground-Truth Q-Functions and Structure

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from typing import Dict, List, Tuple, Callable
import seaborn as sns

sns.set_style('whitegrid')
np.random.seed(42)

## Task 1.1: Implement a Simple Synthetic MDP

### State Structure
The state is composed of 4 blocks: $s = (b_1, b_2, b_3, b_4)$

- $b_1$: real number in $[-1, 1]$
- $b_2$: real number in $[0, 2\pi]$ (periodic/angular)
- $b_3$: binary value in $\{0, 1\}$
- $b_4$: 2D vector $(u, v)$ in $[-1, 1]^2$

### Tasks and Reward Functions
We define 3 tasks, each depending on different subsets of blocks:

- **Task 1**: depends on $b_1$ and $b_2$
  - $r_1(s) = g_1(b_1) + g_2(b_2)$
  - $g_1(x) = x$ (linear)
  - $g_2(y) = \sin(y)$ (periodic)

- **Task 2**: depends only on $b_3$
  - $r_2(s) = g_3(b_3)$
  - $g_3(z) = 2z - 1$ (maps {0,1} to {-1,1})

- **Task 3**: depends on $b_1$ and $b_4$
  - $r_3(s) = g_1(b_1) + g_4(b_4)$
  - $g_1(x) = x$ (same as Task 1)
  - $g_4(u, v) = \sqrt{u^2 + v^2}$ (Euclidean norm)

In [ ]:
class SyntheticMDP:
    """
    A simple synthetic MDP with compositional state structure.
    
    State: s = (b1, b2, b3, b4)
    - b1: scalar in [-1, 1]
    - b2: scalar in [0, 2π] (angular)
    - b3: binary in {0, 1}
    - b4: 2D vector in [-1, 1]²
    """
    
    def __init__(self):
        # Define which blocks each task depends on
        self.task_dependencies = {
            'task1': [1, 2],  # depends on b1 and b2
            'task2': [3],     # depends only on b3
            'task3': [1, 4]   # depends on b1 and b4
        }
        
    def sample_state(self) -> Dict[str, np.ndarray]:
        """
        Sample a random state from the environment.
        
        Returns:
            Dictionary with keys 'b1', 'b2', 'b3', 'b4'
        """
        state = {
            'b1': np.random.uniform(-1, 1),           # scalar in [-1, 1]
            'b2': np.random.uniform(0, 2 * np.pi),    # angular in [0, 2π]
            'b3': np.random.randint(0, 2),            # binary {0, 1}
            'b4': np.random.uniform(-1, 1, size=2)    # 2D vector in [-1,1]²
        }
        return state
    
    # Ground truth component functions
    def g1(self, b1: float) -> float:
        """Linear function for b1"""
        return b1
    
    def g2(self, b2: float) -> float:
        """Periodic function for b2"""
        return np.sin(b2)
    
    def g3(self, b3: int) -> float:
        """Binary mapping for b3: {0,1} -> {-1,1}"""
        return 2 * b3 - 1
    
    def g4(self, b4: np.ndarray) -> float:
        """Euclidean norm for b4"""
        return np.linalg.norm(b4)
    
    def compute_reward(self, state: Dict[str, np.ndarray], task: str) -> float:
        """
        Compute the reward for a given state and task.
        
        Args:
            state: Dictionary with state blocks
            task: Task name ('task1', 'task2', or 'task3')
            
        Returns:
            Reward value (deterministic)
        """
        if task == 'task1':
            # r1(s) = g1(b1) + g2(b2)
            return self.g1(state['b1']) + self.g2(state['b2'])
        
        elif task == 'task2':
            # r2(s) = g3(b3)
            return self.g3(state['b3'])
        
        elif task == 'task3':
            # r3(s) = g1(b1) + g4(b4)
            return self.g1(state['b1']) + self.g4(state['b4'])
        
        else:
            raise ValueError(f"Unknown task: {task}")
    
    def get_task_dependencies(self, task: str) -> List[int]:
        """Return which blocks the task depends on"""
        return self.task_dependencies[task]

### Test the Environment

In [ ]:
# Create environment
env = SyntheticMDP()

# Sample some states and compute rewards
print("=" * 80)
print("Testing Synthetic MDP Environment")
print("=" * 80)

for i in range(5):
    print(f"\n--- Sample {i+1} ---")
    state = env.sample_state()
    
    print(f"State:")
    print(f"  b1 = {state['b1']:.4f}")
    print(f"  b2 = {state['b2']:.4f}")
    print(f"  b3 = {state['b3']}")
    print(f"  b4 = [{state['b4'][0]:.4f}, {state['b4'][1]:.4f}]")
    
    print(f"\nRewards:")
    for task in ['task1', 'task2', 'task3']:
        reward = env.compute_reward(state, task)
        deps = env.get_task_dependencies(task)
        print(f"  {task}: {reward:.4f} (depends on blocks {deps})")

## Task 1.2: Ground-Truth Q-Functions and Structure

### Analytical Q-Functions

Since we're using a contextual bandit (one-step MDP with no state transitions), the Q-function equals the expected reward:

$$Q_i(s) = r_i(s)$$

For each task:

**Task 1:**
$$Q_1(s) = Q_1(b_1, b_2, b_3, b_4) = g_1(b_1) + g_2(b_2) = b_1 + \sin(b_2)$$

**Task 2:**
$$Q_2(s) = Q_2(b_1, b_2, b_3, b_4) = g_3(b_3) = 2b_3 - 1$$

**Task 3:**
$$Q_3(s) = Q_3(b_1, b_2, b_3, b_4) = g_1(b_1) + g_4(b_4) = b_1 + \|b_4\|_2$$

### Compositional Structure Table

In [ ]:
# Create a table showing task-block dependencies
structure_table = pd.DataFrame({
    'Task': ['Task 1', 'Task 2', 'Task 3'],
    'Q-Function': [
        'Q₁(s) = b₁ + sin(b₂)',
        'Q₂(s) = 2b₃ - 1',
        'Q₃(s) = b₁ + ||b₄||₂'
    ],
    'Depends on b₁': ['✓', '', '✓'],
    'Depends on b₂': ['✓', '', ''],
    'Depends on b₃': ['', '✓', ''],
    'Depends on b₄': ['', '', '✓'],
    'Relevant Blocks': ['{1, 2}', '{3}', '{1, 4}'],
    'Decomposition': ['Additive', 'Single block', 'Additive']
})

print("\n" + "=" * 80)
print("Ground Truth Compositional Structure")
print("=" * 80)
print(structure_table.to_string(index=False))
print("\n")

### Implications for Representation Learning

**Key Insights:**

1. **Irrelevant blocks contain no information**: 
   - Task 1 does not need $b_3$ or $b_4$ to predict its reward
   - Task 2 does not need $b_1$, $b_2$, or $b_4$
   - Task 3 does not need $b_2$ or $b_3$

2. **Modular representation is sufficient**:
   - We can learn separate modules $m_1, m_2, m_3, m_4$ for each block
   - Each task only needs to use a subset of these modules
   - This is more efficient than learning a monolithic representation

3. **Correct architecture for each task**:
   - **Task 1**: Use modules $m_1$ and $m_2$ only
   - **Task 2**: Use module $m_3$ only
   - **Task 3**: Use modules $m_1$ and $m_4$ only

4. **Why this matters**:
   - Using irrelevant blocks adds noise and increases sample complexity
   - A modular architecture can share representations across tasks (e.g., $m_1$ is shared between Task 1 and Task 3)
   - The "correct" architecture matches the ground truth causal structure

### Empirical Verification

Let's verify the decomposition numerically by fitting simple regression models.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

# Generate dataset
n_samples = 1000
states = [env.sample_state() for _ in range(n_samples)]

# Extract features and compute rewards
def extract_features(states):
    """Extract all block features from states"""
    b1 = np.array([s['b1'] for s in states])
    b2 = np.array([s['b2'] for s in states])
    b3 = np.array([s['b3'] for s in states])
    b4 = np.array([s['b4'] for s in states])
    
    # Create feature matrix with nonlinear transformations
    features = np.column_stack([
        b1,                    # b1
        np.sin(b2),           # sin(b2)
        np.cos(b2),           # cos(b2)
        b3,                    # b3
        b4[:, 0],             # b4_u
        b4[:, 1],             # b4_v
        np.linalg.norm(b4, axis=1)  # ||b4||
    ])
    
    feature_names = ['b1', 'sin(b2)', 'cos(b2)', 'b3', 'b4_u', 'b4_v', '||b4||']
    return features, feature_names

X, feature_names = extract_features(states)

# Compute rewards for each task
rewards = {
    'task1': np.array([env.compute_reward(s, 'task1') for s in states]),
    'task2': np.array([env.compute_reward(s, 'task2') for s in states]),
    'task3': np.array([env.compute_reward(s, 'task3') for s in states])
}

print("\n" + "=" * 80)
print("Empirical Verification: Linear Regression Analysis")
print("=" * 80)

# Fit linear models for each task
for task_name, y in rewards.items():
    model = LinearRegression()
    model.fit(X, y)
    
    print(f"\n{task_name.upper()}:")
    print(f"R² score: {model.score(X, y):.6f}")
    print(f"\nCoefficients:")
    
    coef_df = pd.DataFrame({
        'Feature': feature_names,
        'Coefficient': model.coef_,
        'Abs Coefficient': np.abs(model.coef_)
    }).sort_values('Abs Coefficient', ascending=False)
    
    print(coef_df.to_string(index=False))
    
    # Highlight relevant features (|coef| > 0.01)
    relevant = coef_df[coef_df['Abs Coefficient'] > 0.01]['Feature'].tolist()
    print(f"\nRelevant features (|coef| > 0.01): {relevant}")

### Visualization: Reward Decomposition

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Task 1: depends on b1 and b2
b1_vals = np.array([s['b1'] for s in states])
b2_vals = np.array([s['b2'] for s in states])

axes[0, 0].scatter(b1_vals, rewards['task1'], alpha=0.3, s=10)
axes[0, 0].set_xlabel('b₁')
axes[0, 0].set_ylabel('Task 1 Reward')
axes[0, 0].set_title('Task 1 vs b₁ (relevant)')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].scatter(b2_vals, rewards['task1'], alpha=0.3, s=10)
axes[0, 1].set_xlabel('b₂')
axes[0, 1].set_ylabel('Task 1 Reward')
axes[0, 1].set_title('Task 1 vs b₂ (relevant)')
axes[0, 1].grid(True, alpha=0.3)

b3_vals = np.array([s['b3'] for s in states])
axes[0, 2].scatter(b3_vals + np.random.normal(0, 0.02, len(b3_vals)), 
                   rewards['task1'], alpha=0.3, s=10)
axes[0, 2].set_xlabel('b₃')
axes[0, 2].set_ylabel('Task 1 Reward')
axes[0, 2].set_title('Task 1 vs b₃ (irrelevant)')
axes[0, 2].grid(True, alpha=0.3)

# Task 2: depends only on b3
axes[1, 0].scatter(b1_vals, rewards['task2'], alpha=0.3, s=10)
axes[1, 0].set_xlabel('b₁')
axes[1, 0].set_ylabel('Task 2 Reward')
axes[1, 0].set_title('Task 2 vs b₁ (irrelevant)')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].scatter(b3_vals + np.random.normal(0, 0.02, len(b3_vals)), 
                   rewards['task2'], alpha=0.3, s=10)
axes[1, 1].set_xlabel('b₃')
axes[1, 1].set_ylabel('Task 2 Reward')
axes[1, 1].set_title('Task 2 vs b₃ (relevant)')
axes[1, 1].grid(True, alpha=0.3)

# Task 3: depends on b1 and b4
b4_norm = np.array([np.linalg.norm(s['b4']) for s in states])
axes[1, 2].scatter(b4_norm, rewards['task3'], alpha=0.3, s=10)
axes[1, 2].set_xlabel('||b₄||')
axes[1, 2].set_ylabel('Task 3 Reward')
axes[1, 2].set_title('Task 3 vs ||b₄|| (relevant)')
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('phase1_reward_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nVisualization saved as 'phase1_reward_decomposition.png'")

## Summary: Phase 1 Deliverables

### ✅ Task 1.1 Completed
- Implemented `SyntheticMDP` environment class
- State structure: $s = (b_1, b_2, b_3, b_4)$ with different types
- Three tasks with deterministic reward functions
- Each task depends on a different subset of blocks

### ✅ Task 1.2 Completed
- Explicit Q-functions written analytically
- Compositional structure table showing task-block dependencies
- Explanation of implications for representation learning
- Empirical verification using linear regression
- Visualization showing relevant vs irrelevant blocks

### Key Takeaways
1. **Blocks** are independent components of the state
2. **Tasks** have different reward functions depending on different blocks
3. **Structure** refers to which blocks each task depends on
4. **Correct architecture** means using only the relevant blocks for each task

### Next Steps
Proceed to **Phase 2**: Implement fixed architectures (monolithic baseline, hand-designed modular, and wrong structure ablation) to see why architecture matters before doing any search.